# 🍽️ Restaurant Chain Analysis
## Identifying & Profiling Restaurant Chains — Rating & Popularity Study

> **Objective:** Identify restaurant chains within the dataset, quantify their presence, and conduct a thorough analysis of their ratings, geographic spread, popularity metrics, and competitive positioning relative to standalone restaurants.

---

**Dataset:** Global Restaurant Dataset | **Records:** 9,551 | **Features:** 21

## 1. Environment Setup & Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Global Styling ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f9f9f9',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('✅ Libraries loaded successfully.')

## 2. Data Loading & Initial Exploration

In [ ]:
# Load dataset
df = pd.read_csv('../data/dataset.csv')

print(f'Shape         : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns       : {list(df.columns)}')
print(f'\nNull values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

df.head()

In [ ]:
# Descriptive statistics for numerical columns
df[['Aggregate rating', 'Votes', 'Average Cost for two', 'Price range']].describe().round(2)

## 3. Identifying Restaurant Chains

> A **restaurant chain** is defined as any brand name appearing at **two or more distinct outlet entries** in the dataset. This is the standard operational definition used in the food service industry.

In [ ]:
# Count frequency of each restaurant name
name_counts = df['Restaurant Name'].value_counts()

# Chains = names with 2+ entries
chain_names = name_counts[name_counts > 1].index
chain_mask  = df['Restaurant Name'].isin(chain_names)

chains_df     = df[chain_mask].copy()
standalone_df = df[~chain_mask].copy()

print('━' * 55)
print(f'  Total records             : {len(df):>7,}')
print(f'  Unique restaurant names   : {df["Restaurant Name"].nunique():>7,}')
print(f'  Distinct chains detected  : {len(chain_names):>7,}')
print(f'  Chain outlet records      : {len(chains_df):>7,}  ({chain_mask.mean()*100:.1f}%)')
print(f'  Standalone restaurants    : {len(standalone_df):>7,}  ({(~chain_mask).mean()*100:.1f}%)')
print('━' * 55)

In [ ]:
# Build comprehensive chain summary table
chain_summary = (
    df.groupby('Restaurant Name')
      .agg(
          outlet_count   = ('Restaurant Name', 'count'),
          avg_rating     = ('Aggregate rating', 'mean'),
          total_votes    = ('Votes', 'sum'),
          avg_votes      = ('Votes', 'mean'),
          cities_covered = ('City', 'nunique'),
          countries      = ('Country Code', 'nunique'),
          avg_cost       = ('Average Cost for two', 'mean'),
      )
      .query('outlet_count > 1')
      .sort_values('outlet_count', ascending=False)
      .reset_index()
)
chain_summary['avg_rating'] = chain_summary['avg_rating'].round(2)
chain_summary['avg_votes']  = chain_summary['avg_votes'].round(1)
chain_summary['avg_cost']   = chain_summary['avg_cost'].round(0)

print(f'Chain summary table built: {len(chain_summary)} chains')
chain_summary.head(15)

## 4. Chain Distribution Analysis
### 4.1 Top 15 Chains by Number of Outlets

In [ ]:
top15 = chain_summary.head(15)

fig, ax = plt.subplots(figsize=(12, 7))
colors = sns.color_palette('Blues_d', len(top15))[::-1]
bars = ax.barh(top15['Restaurant Name'][::-1], top15['outlet_count'][::-1],
               color=colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, top15['outlet_count'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10, fontweight='bold', color='#2E4057')
ax.set_xlabel('Number of Outlets', labelpad=10)
ax.set_title('Top 15 Restaurant Chains by Number of Outlets', pad=15)
ax.set_xlim(0, top15['outlet_count'].max() * 1.12)
plt.tight_layout()
plt.savefig('../outputs/fig1_top15_chains_by_outlets.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

## 5. Rating Analysis
### 5.1 Chains vs Standalone — Rating Comparison

In [ ]:
rated_chains     = chains_df[chains_df['Aggregate rating'] > 0]['Aggregate rating']
rated_standalone = standalone_df[standalone_df['Aggregate rating'] > 0]['Aggregate rating']

print('Rating Statistics — Chains vs Standalone')
print('━' * 50)
print(f'  Metric               Chains    Standalone')
print(f'  Mean Rating        : {rated_chains.mean():.3f}     {rated_standalone.mean():.3f}')
print(f'  Median Rating      : {rated_chains.median():.3f}     {rated_standalone.median():.3f}')
print(f'  Std Deviation      : {rated_chains.std():.3f}     {rated_standalone.std():.3f}')
print(f'  Min Rating         : {rated_chains.min():.1f}       {rated_standalone.min():.1f}')
print(f'  Max Rating         : {rated_chains.max():.1f}       {rated_standalone.max():.1f}')
print('━' * 50)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(rated_chains, bins=30, alpha=0.65, color='#048A81',
             label=f'Chains (n={len(rated_chains):,})', density=True, edgecolor='white')
axes[0].hist(rated_standalone, bins=30, alpha=0.65, color='#EF946C',
             label=f'Standalone (n={len(rated_standalone):,})', density=True, edgecolor='white')
axes[0].axvline(rated_chains.mean(), color='#048A81', linestyle='--', linewidth=2,
                label=f'Chain Mean: {rated_chains.mean():.2f}')
axes[0].axvline(rated_standalone.mean(), color='#EF946C', linestyle='--', linewidth=2,
                label=f'Standalone Mean: {rated_standalone.mean():.2f}')
axes[0].set_xlabel('Aggregate Rating')
axes[0].set_ylabel('Density')
axes[0].set_title('Rating Distribution: Chains vs Standalone')
axes[0].legend(fontsize=9)

bp = axes[1].boxplot([rated_chains.values, rated_standalone.values],
                     patch_artist=True, labels=['Chains', 'Standalone'],
                     medianprops=dict(color='white', linewidth=2))
bp['boxes'][0].set_facecolor('#048A81')
bp['boxes'][1].set_facecolor('#EF946C')
axes[1].set_ylabel('Aggregate Rating')
axes[1].set_title('Rating Spread: Chains vs Standalone')

plt.suptitle('Rating Analysis — Chains vs Standalone Restaurants', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/fig2_rating_chains_vs_standalone.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

## 6. Popularity & Engagement
### 6.1 Bubble Chart — Outlets vs Rating vs Votes

In [ ]:
top20 = chain_summary[chain_summary['avg_rating'] > 0].nlargest(20, 'outlet_count')

fig, ax = plt.subplots(figsize=(13, 7))
scatter = ax.scatter(
    top20['outlet_count'],
    top20['avg_rating'],
    s=top20['total_votes'] / top20['total_votes'].max() * 2000 + 100,
    c=top20['avg_rating'],
    cmap='RdYlGn', alpha=0.8, edgecolors='grey', linewidth=0.6,
    vmin=2, vmax=5
)
for _, row in top20.iterrows():
    ax.annotate(row['Restaurant Name'], (row['outlet_count'], row['avg_rating']),
                textcoords='offset points', xytext=(5, 5), fontsize=7.5, color='#2E4057')
plt.colorbar(scatter, ax=ax, label='Avg Rating')
ax.set_xlabel('Number of Outlets')
ax.set_ylabel('Average Aggregate Rating')
ax.set_title('Chain Popularity vs Rating\n(Bubble size = Total Votes)', pad=12)
plt.tight_layout()
plt.savefig('../outputs/fig3_chain_popularity_bubble.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

## 7. Rating Category Breakdown — Top 10 Chains

In [ ]:
top10_names  = chain_summary.head(10)['Restaurant Name'].tolist()
rating_order = ['Excellent', 'Very Good', 'Good', 'Average', 'Poor', 'Not rated']
rating_colors= ['#2a9d8f', '#57cc99', '#90e0ef', '#f4a261', '#e76f51', '#adb5bd']

top10_df = df[df['Restaurant Name'].isin(top10_names)]
pivot    = top10_df.groupby(['Restaurant Name', 'Rating text']).size().unstack(fill_value=0)
pivot    = pivot.reindex(columns=[c for c in rating_order if c in pivot.columns])
pivot_pct= pivot.div(pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 7))
pivot_pct.plot(kind='bar', stacked=True, ax=ax,
               color=rating_colors[:len(pivot_pct.columns)], edgecolor='white', linewidth=0.5)
ax.set_xlabel('Restaurant Chain', labelpad=10)
ax.set_ylabel('Percentage of Outlets (%)')
ax.set_title('Rating Category Distribution — Top 10 Chains', pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=9)
ax.legend(title='Rating Category', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/fig4_rating_category_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

## 8. Geographic Spread — Top 10 Chains

In [ ]:
geo_df = chain_summary.head(10)[['Restaurant Name', 'cities_covered', 'outlet_count']]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(geo_df))
w = 0.35
ax.bar(x - w/2, geo_df['cities_covered'], width=w, label='Cities', color='#2E4057', alpha=0.85)
ax.bar(x + w/2, geo_df['outlet_count'],   width=w, label='Outlets', color='#048A81', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(geo_df['Restaurant Name'], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Count')
ax.set_title('Geographic Spread — Cities Covered vs Outlet Count (Top 10 Chains)', pad=12)
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/fig5_geographic_spread.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 5 saved.')

## 9. Votes Analysis — Total & Per-Outlet

In [ ]:
top15v  = chain_summary.nlargest(15, 'total_votes')
top15av = chain_summary.nlargest(15, 'avg_votes')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
colors_tv = sns.color_palette('Blues_d', len(top15v))
axes[0].barh(top15v['Restaurant Name'][::-1], top15v['total_votes'][::-1],
             color=colors_tv[::-1], edgecolor='white')
axes[0].set_xlabel('Total Votes')
axes[0].set_title('Total Votes — Top 15 Chains')

colors_av = sns.color_palette('Greens_d', len(top15av))
axes[1].barh(top15av['Restaurant Name'][::-1], top15av['avg_votes'][::-1],
             color=colors_av[::-1], edgecolor='white')
axes[1].set_xlabel('Avg Votes per Outlet')
axes[1].set_title('Avg Votes per Outlet — Top 15 Chains')

plt.suptitle('Popularity (Votes) Analysis — Restaurant Chains', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/fig6_votes_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 6 saved.')

## 10. Performance Heatmap — Top 15 Chains

In [ ]:
hm_df = chain_summary.head(15).set_index('Restaurant Name')[
    ['outlet_count', 'avg_rating', 'avg_votes', 'cities_covered']
].copy()
hm_norm = (hm_df - hm_df.min()) / (hm_df.max() - hm_df.min())
hm_norm.columns = ['Outlets', 'Avg Rating', 'Avg Votes', 'Cities Covered']

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(hm_norm, annot=hm_df.round(1), fmt='g', cmap='YlGnBu', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Normalized Score'})
ax.set_title('Performance Heatmap — Top 15 Chains\n(Color = Normalized Score, Value = Actual)', pad=12)
plt.tight_layout()
plt.savefig('../outputs/fig7_performance_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 7 saved.')

## 11. Key Findings & Summary

In [ ]:
top_rated_chains = chain_summary[chain_summary['outlet_count'] >= 5].nlargest(5, 'avg_rating')
most_popular     = chain_summary.nlargest(5, 'total_votes')

print('=' * 60)
print('           ANALYSIS SUMMARY — KEY FINDINGS')
print('=' * 60)
print(f'  Total Restaurants Analyzed  : {len(df):,}')
print(f'  Restaurant Chains Detected  : {len(chain_names):,}')
print(f'  Chain Outlet Coverage       : {chain_mask.mean()*100:.1f}% of dataset')
print(f'  Standalone Restaurants      : {(~chain_mask).sum():,}')
print()
print('  ── Rating Insights ──')
print(f'  Avg Rating (Chains)         : {rated_chains.mean():.3f}')
print(f'  Avg Rating (Standalone)     : {rated_standalone.mean():.3f}')
print()
print('  ── Top 5 Highest-Rated Chains (min 5 outlets) ──')
for _, row in top_rated_chains.iterrows():
    print(f'  {row["Restaurant Name"]:<25} Avg: {row["avg_rating"]:.2f}  Outlets: {row["outlet_count"]}')
print()
print('  ── Top 5 Most Voted Chains ──')
for _, row in most_popular.iterrows():
    print(f'  {row["Restaurant Name"]:<25} Total Votes: {row["total_votes"]:,}')
print('=' * 60)
print('\n✅ Analysis complete. All outputs saved to /outputs/')